<table border=0 width="100%"><tr><td><p align="left"><img src="..\img\logo.png" align="left" width=300></p></td><td><font size=3><B>Lecture 6 实验 （郑海超）</B></font></td></tr></table>

# 实验要求
- 读取上证指数股吧的评论（sse_comments.txt）-2025年10月20日的部分评论;
- 使用我提供的情感词表来对每条评论做情感分析；
- 统计汇总所有的情感分析结果
- 作业完成过程的经验总结


**提交作业：**
- 在线提交：http://swufe.fanya.chaoxing.com/portal
- 截止日期：2025年10月29日晚上8点
- 仅仅提交**这个jupyter文档**，命名为“lecture6_experiment_你的学号+姓名.ipynb”，其中学号和姓名请替换为你自己的学号和姓名。


**下周我们邀请同学分享其完成作业的结果和过程经验总结**

# 数据读取
- sse_comments.txt

In [6]:
# 从sse_comments.txt读取数据
"""
直接读取上证指数股吧评论数据的脚本
"""

import os

# 获取当前脚本所在目录
current_dir = os.path.dirname(os.path.abspath("sse_comments.txt"))
# 拼接评论文件的完整路径
file_path = os.path.join(current_dir, 'sse_comments.txt')

# 读取并过滤评论数据
comments = []
oct_20_comments = []
try:
    # 使用utf-8编码读取文件
    with open(file_path, 'r', encoding='utf-8') as f:
        # 逐行读取并处理
        for line in f:
            # 去除首尾空白字符
            comment = line.strip()
            # 只添加非空行
            if comment:
                comments.append(comment)
                # 过滤出包含10月20日的评论
                if '10月20日' in comment or '10-20' in comment or '10/20' in comment:
                    oct_20_comments.append(comment)
    
    # 打印读取结果信息
    print(f"成功读取 {len(comments)} 条评论")
    print(f"10月20日的评论数量: {len(oct_20_comments)}")
    
    # 显示10月20日评论的前5条作为示例
    print("\n10月20日评论示例：")
    for i, comment in enumerate(oct_20_comments[:5], 1):
        print(f"{i}. {comment}")
        
except FileNotFoundError:
    print(f"错误：找不到文件 {file_path}")
except Exception as e:
    print(f"读取文件时出错：{str(e)}")

# 现在可以直接使用oct_20_comments变量进行后续的情感分析处理

成功读取 401 条评论
10月20日的评论数量: 400

10月20日评论示例：
1. 2025-10-20 15:36:39 (本想退出，奈何真不甘心，平台新到一些先X后B，月还五K，以稳为主，再玩一段时间
2. 2025-10-20 15:36:39 多头整理行情结束的标志是什么？
3. 2025-10-20 15:36:38 2025.10.20周一（0093）等放量下跌，才会有修复！
4. 2025-10-20 15:36:36 明天上午还有一次冲击的机会，过不去可以全部跑
5. 2025-10-20 15:36:23 礼拜五拿我三千今天还我一千怎么玩


# 写一个针对每条股吧评论的情感分类器：Naive_Classifier 函数
- 函数的参数是一个字符串，即一条股吧评论
- 函数的返回值是一个整数，即情感分类结果（-1表示负面情感，1表示正面情感，0表示中性情感）

函数的实现步骤如下：

> 情感词典中的每个词语都被标注为正面或负面（正面词典和负面词典）；

> 统计评论中出现正面词汇和负面词汇的总个数，如果重复出现，就多次统计上；

> 计算所有匹配词典词语的净数量（正面词汇个数-负面词汇个数）：若该值大于1，返回1；若该值小于-1，返回-1；否则返回0

**情感词典**
- zhang_unformal_neg.txt
- zhang_unformal_pos.txt

停用词词典（如果使用切词，考虑使用这个停用词词典）
- userdict.txt

去掉pass，开始编程

In [9]:
def naive_classifier(single_comment: str):
    """
    对一条评论做情感分析
    
    参数:
        single_comment(str): 评论文本
    
    返回:
        int: 情感分析结果，返回 0,1，或者-1
    """
    # 开始编程

   # 情感分类器函数实现

# 在模块级别加载词典，避免重复加载
pos_words = set()
neg_words = set()
user_dict = set()

# 读取正面词典
try:
    with open('zhang_unformal_pos.txt', 'r', encoding='utf-8') as f:
        for line in f:
            word = line.strip()
            if word:
                pos_words.add(word)
    print(f"成功加载正面词典，共 {len(pos_words)} 个词汇")
except FileNotFoundError:
    print("警告: 正面词典文件未找到，使用默认正面词汇")
    pos_words = {'上涨', '大涨', '涨停', '反弹', '利好', '强势', '突破', '企稳', '回升', '看好'}

# 读取负面词典
try:
    with open('zhang_unformal_neg.txt', 'r', encoding='utf-8') as f:
        for line in f:
            word = line.strip()
            if word:
                neg_words.add(word)
    print(f"成功加载负面词典，共 {len(neg_words)} 个词汇")
except FileNotFoundError:
    print("警告: 负面词典文件未找到，使用默认负面词汇")
    neg_words = {'下跌', '大跌', '跌停', '暴跌', '利空', '弱势', '破位', '回调', '回落', '看空',
                 '套', '亏损', '亏', '跌', '绿', '熊', '弱', '崩', '闷杀', '诱多', '割肉', '雷', '撤'}

# 读取用户词典
try:
    with open('userdict.txt', 'r', encoding='utf-8') as f:
        for line in f:
            word = line.strip()
            if word:
                user_dict.add(word)
    print(f"成功加载用户词典，共 {len(user_dict)} 个词汇")
except FileNotFoundError:
    print("警告: 用户词典文件未找到，将不使用用户词典")

def naive_classifier(single_comment: str):
    """
    对单条股吧评论进行情感分析
    
    参数:
        single_comment(str): 评论文本
    
    返回:
        int: 情感分析结果，返回 -1(负面), 0(中性), 或 1(正面)
    """
    # 统计正面词汇出现次数
    pos_count = 0
    for word in pos_words:
        pos_count += single_comment.count(word)
    
    # 统计负面词汇出现次数
    neg_count = 0
    for word in neg_words:
        neg_count += single_comment.count(word)
    
    # 计算净数量
    net_score = pos_count - neg_count
    
    # 根据净数量判断情感类别
    if net_score > 1:
        return 1  # 正面情感
    elif net_score < -1:
        return -1  # 负面情感
    else:
        return 0  # 中性情感

# 测试代码示例
if __name__ == "__main__":
    # 准备测试评论
    test_comments = []
    
    # 尝试从文件读取评论
    try:
        with open('sse_comments.txt', 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if line.strip() and i < 5:  # 取前5条非空评论
                    test_comments.append(line.strip())
    except FileNotFoundError:
        # 如果文件不存在，使用预设评论
        test_comments = [
            "今天市场大幅上涨，迎来强劲反弹！",
            "股票又跌了，真是让人失望。",
            "市场表现平平，没有明显方向。",
            "重大利好消息，明天肯定大涨！",
            "暴跌收场，亏惨了。"
        ]
    
    # 一次性处理并显示所有5条评论的结果
    print("\n===== 情感分析测试结果 =====")
    for i, comment in enumerate(test_comments, 1):
        result = naive_classifier(comment)
        sentiment = "正面" if result == 1 else "负面" if result == -1 else "中性"
        print(f"评论 {i}: {comment}")
        print(f"情感分析结果: {result} ({sentiment})")
        print("---")
    


成功加载正面词典，共 911 个词汇
成功加载负面词典，共 964 个词汇
成功加载用户词典，共 7510 个词汇

===== 情感分析测试结果 =====
评论 1: 2025-10-17 17:41:18 收市盘点：市场震荡反弹，能企稳回暖吗？
情感分析结果: 1 (正面)
---
评论 2: 2025-10-20 15:36:39 (本想退出，奈何真不甘心，平台新到一些先X后B，月还五K，以稳为主，再玩一段时间
情感分析结果: 0 (中性)
---
评论 3: 2025-10-20 15:36:39 多头整理行情结束的标志是什么？
情感分析结果: 0 (中性)
---
评论 4: 2025-10-20 15:36:38 2025.10.20周一（0093）等放量下跌，才会有修复！
情感分析结果: -1 (负面)
---
评论 5: 2025-10-20 15:36:36 明天上午还有一次冲击的机会，过不去可以全部跑
情感分析结果: 0 (中性)
---


# 汇总所有评论的情感分析结果
- 应用上述函数naive_classifier，对所有评论做情感分析
- 对情感分析结果进行汇总，统计正面评论和负面评论的数量，以及总评论数量
- 计算正面评论占比和负面评论占比

In [12]:
# 应用上述函数naive_classifier，对所有评论做情感分析

# 调用已写好的naive_classifier函数分析评论

# 存储每条评论的分析结果
results = []

# 调用分类器分析所有评论
for comment in oct_20_comments:
    sentiment = naive_classifier(comment)
    results.append((comment, sentiment))


In [21]:
# 对所有评论的情感分析结果进行汇总，统计正面评论和负面评论的数量，以及总评论数量
# 统计各类情感数量
positive_count = sum(1 for _, s in results if s == 1)
negative_count = sum(1 for _, s in results if s == -1)
neutral_count = sum(1 for _, s in results if s == 0)
total_count = len(results)


# 清晰展示统计结果
print("=" * 50)
print("评论情感分析统计结果")
print("=" * 50)
print(f"总评论数: {total_count}")
print(f"正面评论: {positive_count} 条")
print(f"负面评论: {negative_count} 条 ")
print(f"中性评论: {neutral_count} 条 ")
print("=" * 50)

# 按每组5条展示部分评论的分析结果
print("\n评论分析示例:")
print("-" * 50)

for i in range(0, min(15, len(results)), 5):  # 展示最多15条评论，每组5条
    group = results[i:i+5]
    for idx, (comment, sentiment) in enumerate(group, 1):
        sentiment_label = "正面" if sentiment == 1 else "负面" if sentiment == -1 else "中性"
        print(f"评论{i+idx}: {comment[:50]}...")
        print(f"情感: {sentiment_label} ({sentiment})")
    print("-" * 50)



评论情感分析统计结果
总评论数: 400
正面评论: 47 条
负面评论: 35 条 
中性评论: 318 条 

评论分析示例:
--------------------------------------------------
评论1: 2025-10-20 15:36:39 (本想退出，奈何真不甘心，平台新到一些先X后B，月还五K，以...
情感: 中性 (0)
评论2: 2025-10-20 15:36:39 多头整理行情结束的标志是什么？...
情感: 中性 (0)
评论3: 2025-10-20 15:36:38 2025.10.20周一（0093）等放量下跌，才会有修复！...
情感: 负面 (-1)
评论4: 2025-10-20 15:36:36 明天上午还有一次冲击的机会，过不去可以全部跑...
情感: 中性 (0)
评论5: 2025-10-20 15:36:23 礼拜五拿我三千今天还我一千怎么玩...
情感: 中性 (0)
--------------------------------------------------
评论6: 2025-10-20 15:36:06 虽说红开后大抵随分时的反抽走修复，但是修复力度的强弱还是得客...
情感: 中性 (0)
评论7: 2025-10-20 15:35:45 今日盘中五次拉高诱多韭菜花入套...
情感: 中性 (0)
评论8: 2025-10-20 15:35:42 转熊了，搞个luan！...
情感: 中性 (0)
评论9: 2025-10-20 15:35:33 【10月21日上证指数预测】微盘股明天有望突破，沪指次之，创...
情感: 正面 (1)
评论10: 2025-10-20 15:35:23 收评|A股全线上涨！历史重演！又爆了！...
情感: 正面 (1)
--------------------------------------------------
评论11: 2025-10-20 15:35:04 重磅利好！特朗普释放缓和信号，科技股全线反弹了，CPO算力强...
情感: 正面 (1)
评论12: 2025-10-20 15:34:52 虽然红了，感觉有点假，管住手啊...
情感: 中性 (0)
评论13: 2025-10-20 15:34:40 马上要万马奔腾.

In [22]:
# 计算正面评论占比和负面评论占比
# 计算各类情感占比
positive_ratio = positive_count / total_count * 100
negative_ratio = negative_count / total_count * 100
neutral_ratio = neutral_count / total_count * 100

# 额外展示统计信息的可视化（文本形式）
print("\n情感分布概览:")
print(f"正面: {'█' * int(positive_ratio/5):<20} {positive_ratio:.1f}%")
print(f"负面: {'█' * int(negative_ratio/5):<20} {negative_ratio:.1f}%")
print(f"中性: {'█' * int(neutral_ratio/5):<20} {neutral_ratio:.1f}%")




情感分布概览:
正面: ██                   11.8%
负面: █                    8.8%
中性: ███████████████      79.5%


# 作业完成过程的经验总结
- 实验过程中遇到的问题和解决方法

 > 首先在读取数据的时候，忘记筛选10月20号的评论，后来重新编写代码，加入了筛选条件

 > 在读取正负面字典的同时利用大模型进行了一些补充，并且提供了读取失败的处理机制

 > 在写情感分类器函数的时候，trae给的代码含有重复读取的步骤，我进行优化，在模块级别加载词典，避免了重复加载。

 > 刚开始觉得按照老师给的正负面词典得到的结果不太准确，中性的结果太多了，但是不知道除了完善词典库还可以如何改善，所以没有进行优化